<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0 |

</details>


# Model Editing

Interventions inside `model.trace()` are temporary — they only apply during that single forward pass. With `model.edit()`, you can create **persistently modified** versions of a model whose interventions replay on every subsequent forward pass.

## Setup

In [1]:
import torch
import torch.nn as nn
from nnsight import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Creating an Edited Model

Use `model.edit()` to define persistent interventions. By default (`inplace=False`) it stores the edit on a **shallow copy** of the model — the original is left clean. Entering the block binds a `(tracer, edited)` tuple: write your interventions against `edited`, and later trace through `edited` to replay them.

In [2]:
# First, capture the last-layer hidden states that produce "Paris"
with model.trace("The Eiffel Tower is in the city of"):
    paris_hs = model.transformer.h[-1].output[:, -1, :].save()

# Create an edited model that always injects the "Paris" hidden states at the last layer
with model.edit() as (tracer, edited):
    edited.transformer.h[-1].output[:, -1, :] = paris_hs

# Original model still works normally
with model.trace("Vatican is in the city of"):
    original = model.lm_head.output.argmax(dim=-1).save()

# Edited model always predicts "Paris"
with edited.trace("Vatican is in the city of"):
    modified = edited.lm_head.output.argmax(dim=-1).save()

print(f"Original:  {model.tokenizer.decode(original[0, -1])}")
print(f"Edited:    {edited.tokenizer.decode(modified[0, -1])}")

Original:   Rome
Edited:     Paris


The edit persists across every trace through the edited model:

In [3]:
prompts = [
    "The Colosseum is in the city of",
    "Big Ben is in the city of",
    "The Statue of Liberty is in the city of",
]

for prompt in prompts:
    with edited.trace(prompt):
        tokens = edited.lm_head.output.argmax(dim=-1).save()
    print(f"{prompt} → {edited.tokenizer.decode(tokens[0, -1])}")

The Colosseum is in the city of →  Paris
Big Ben is in the city of →  Paris
The Statue of Liberty is in the city of →  Paris


<details class="admonition note">
<summary>How editing works</summary>

`model.edit()` captures your interventions and stores them on the envoy instead of running them once. On every later trace, the stored edits replay **first** — before your invoke's own interventions — so their effects are visible to the rest of your trace. The edit context uses the same syntax as `model.trace()`; you access `.output` and `.input` on modules the same way.

Non-inplace `edit()` stores the edit on a shallow copy (`edited`). The underlying `torch.nn.Module`, interleaver, and children are shared with `model` — only the list of stored edits is independent — so no weights are duplicated.

</details>

## In-Place Editing

By default, `model.edit()` leaves the original untouched. To modify the original model directly, pass `inplace=True`. The block then binds only the `tracer` (there is no separate `edited` copy), and you write against `model` itself.

In [4]:
with model.edit(inplace=True) as tracer:
    model.transformer.h[-1].output[:, -1, :] = paris_hs

# Now the original model itself is edited — every trace replays the edit
with model.trace("Vatican is in the city of"):
    tokens = model.lm_head.output.argmax(dim=-1).save()

print(f"In-place edited: {model.tokenizer.decode(tokens[0, -1])}")

In-place edited:  Paris


<details class="admonition warning">
<summary>Use inplace=True with caution</summary>

In-place edits affect **all** subsequent forward passes through the model, including every `model.trace()` call. If you're experimenting, prefer the default (non-inplace) mode so the original model stays clean.

</details>

## Clearing Edits

Use `.clear_edits()` to drop all stored edits and restore the model to its original behavior.

In [5]:
model.clear_edits()

with model.trace("Vatican is in the city of"):
    tokens = model.lm_head.output.argmax(dim=-1).save()

print(f"After clear_edits(): {model.tokenizer.decode(tokens[0, -1])}")

After clear_edits():  Rome


## Attaching Custom Modules

You can attach your own PyTorch modules to the model — like an SAE or LoRA adapter — and wire them into the forward pass with `model.edit()`. Calling the attached module with `hook=True` runs its full `__call__`, so its submodules become **fully instrumented**: you can access its `.output` in a trace just like any other module.

In [6]:
# Define a low-rank adapter
class Adapter(nn.Module):
    def __init__(self, dim, rank=16):
        super().__init__()
        self.down = nn.Linear(dim, rank, bias=False)
        self.up = nn.Linear(rank, dim, bias=False)
        nn.init.zeros_(self.up.weight)  # Start as identity (no effect)

    def forward(self, x):
        return self.up(self.down(x))

# Attach it to layer 5 (on the same device as the layer)
device = next(model.transformer.h[5]._module.parameters()).device
model.transformer.h[5].adapter = Adapter(768, rank=16).to(device)

# Wire the adapter into the forward pass: layer output += adapter(layer input)
with model.edit() as (tracer, edited):
    h5_input = edited.transformer.h[5].inputs[0][0]
    adapter_out = edited.transformer.h[5].adapter(h5_input, hook=True)
    edited.transformer.h[5].output[:] = edited.transformer.h[5].output + adapter_out

With zero-initialized weights, the adapter has no effect:

In [7]:
with model.trace("The Eiffel Tower is in the city of"):
    orig_logits = model.lm_head.output.save()

with edited.trace("The Eiffel Tower is in the city of"):
    adapter_result = edited.transformer.h[5].adapter.output.save()
    edited_logits = edited.lm_head.output.save()

print(f"Adapter output shape: {adapter_result.shape}")
print(f"Adapter norm: {adapter_result.norm():.4f}")
print(f"Original prediction:  {model.tokenizer.decode(orig_logits[0, -1].argmax(dim=-1))}")
print(f"Adapted prediction:   {edited.tokenizer.decode(edited_logits[0, -1].argmax(dim=-1))}")

Adapter output shape: torch.Size([1, 10, 768])
Adapter norm: 0.0000
Original prediction:   Paris
Adapted prediction:    Paris


After training or modifying the adapter weights, it changes the model's behavior:

In [8]:
# Simulate trained weights by setting non-zero values
nn.init.normal_(model.transformer.h[5].adapter._module.up.weight, std=10.0)

with edited.trace("The Eiffel Tower is in the city of"):
    adapter_result = edited.transformer.h[5].adapter.output.save()
    adapted_logits = edited.lm_head.output.save()

print(f"Adapter output norm: {adapter_result.norm():.4f}")
print(f"Adapted prediction:  {edited.tokenizer.decode(adapted_logits[0, -1].argmax(dim=-1))}")

Adapter output norm: 69833.6797


Adapted prediction:   


You can also **intervene on the adapter itself** during a trace — for example, zeroing out its contribution:

In [9]:
with edited.trace("The Eiffel Tower is in the city of"):
    # Zero out the adapter's output, neutralizing it for this run
    edited.transformer.h[5].adapter.output[:] = 0
    logits = edited.lm_head.output.save()

print(f"Adapter zeroed: {edited.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")

Adapter zeroed:  Paris


<details class="admonition tip">
<summary>When to use editing vs tracing</summary>

- Use `model.trace()` for **one-off** interventions during a single forward pass
- Use `model.edit()` when you need the **same intervention applied repeatedly** across many forward passes
- Use `model.edit()` to wire custom modules (adapters, SAEs, probes) into the model's forward pass, making them automatically instrumented for tracing

</details>